# 최소제곱법과 정규방정식 — 완벽한 답이 없을 때 최선의 답 찾기

**Zero to 머신러닝 딥러닝 Master — Medium 시리즈 #05** · 원문: 책 00-03 「연립방정식 — "x가 뭔지 맞춰보세요" 게임」

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/otoloism/Zero_to_ML_Master/blob/main/medium-series/05-least-squares/05_least_squares.ipynb)

식이 미지수보다 많은 과결정계에서 "오차 제곱합이 가장 작은 답"을 찾는 최소제곱법을 다룹니다. 정규방정식 x̂ = (AᵀA)⁻¹Aᵀb 를 그대로 구현한 방법과 실무 표준 `np.linalg.lstsq` 를 키·몸무게 데이터로 비교하고, 잔차·오차제곱합·직교 조건(Aᵀe = 0)까지 검증합니다.

- 📝 Medium 글: [최소제곱법과 정규방정식 — 완벽한 답이 없을 때 최선의 답 찾기](https://medium.com/p/933746f6a6a1)
- 📘 책 원문: [00-03 연립방정식 — "x가 뭔지 맞춰보세요" 게임](https://wikidocs.net/439841)
- 🏠 [mldict.net](https://mldict.net) · ✍️ [기술 블로그](https://wikidocs.net/blog/@mldict/) · 📚 [책 전체 목차](https://wikidocs.net/book/21464)

## 00-03-R2 🔴 (참고) 공식을 손으로 직접 굴려보기 — 데이터 3개 미니 예제

점 (1, 2), (2, 3), (3, 5)를 지나는 최선의 직선 y = a·x + b 를 정규방정식으로 손계산한 과정입니다 (책 본문 그대로, 실행 코드 아님).

```text
Aᵀ = [[1, 2, 3],
      [1, 1, 1]]        ← A의 행과 열을 맞바꾼 것 (2×3)

AᵀA (1행1열) = 1×1 + 2×2 + 3×3 = 14   ← Σx²
AᵀA (1행2열) = 1×1 + 2×1 + 3×1 =  6   ← Σx
AᵀA (2행1열) = 1×1 + 1×2 + 1×3 =  6   ← Σx
AᵀA (2행2열) = 1×1 + 1×1 + 1×1 =  3   ← 데이터 개수 m

AᵀA = [[14, 6],
       [ 6, 3]]         ← 2×2 정사각행렬 완성! ✅
```

```text
Aᵀb (1행) = 1×2 + 2×3 + 3×5 = 2 + 6 + 15 = 23   ← Σxy
Aᵀb (2행) = 1×2 + 1×3 + 1×5 = 2 + 3 +  5 = 10   ← Σy

Aᵀb = [23, 10]
```

```text
행렬식 det = 14×3 - 6×6 = 42 - 36 = 6      ← 0이 아니므로 역행렬 존재 ✅

(AᵀA)⁻¹ = (1/6) × [[ 3, -6],
                    [-6, 14]]
```

```text
a = (1/6) × ( 3×23 + (-6)×10 ) = (1/6) × (69 - 60) = 9/6 = 1.5
b = (1/6) × ((-6)×23 +  14×10 ) = (1/6) × (-138 + 140) = 2/6 ≈ 0.3333

→ 찾은 직선:  y = 1.5x + 0.333
```

## 1. 00-03-R3 🔴 구현 원리와 코드 — 공식·lstsq 두 방법 비교

5명의 키·몸무게 데이터로 정규방정식 공식과 `lstsq` 를 모두 돌려보고, 잔차·오차제곱합·직교 조건을 확인한 뒤 새 키로 몸무게를 예측합니다.

In [1]:
import numpy as np

# ── 데이터: 5명의 키(cm)와 몸무게(kg) ──────────────────────
height = np.array([165, 170, 175, 180, 185], dtype=float)
weight = np.array([58,  65,  70,  75,  82],  dtype=float)

# ── 설계행렬 A 만들기 ─────────────────────────────────────
# 찾고 싶은 직선: weight = a×height + b×1
# → 각 행이 (height, 1)이 되도록 1로 채운 열을 붙입니다 (절편 항!)
A = np.column_stack([height, np.ones(len(height))])
b = weight

print("A shape:", A.shape, "| b shape:", b.shape)   # (5,2) (5,)

# ── 방법 1) 정규방정식 공식을 그대로 구현 (이해용) ──────────
# x̂ = (AᵀA)⁻¹ Aᵀb  ← 방금 유도한 공식 그 자체
x_formula = np.linalg.inv(A.T @ A) @ A.T @ b

# ── 방법 2) lstsq 사용 (실무 권장) ────────────────────────
# rcond=None: 버전 경고 방지 / [0]: 반환 튜플의 첫 원소가 해
x_lstsq = np.linalg.lstsq(A, b, rcond=None)[0]

a_hat, b_hat = x_lstsq          # 튜플 언패킹: 기울기, 절편
print("공식 방식 :", np.round(x_formula, 4))
print("lstsq     :", np.round(x_lstsq,   4))
print(f"찾은 직선: 몸무게 = {a_hat:.2f} × 키 + ({b_hat:.2f})")

# ── 잔차와 오차 제곱합 확인 ───────────────────────────────
pred     = A @ x_lstsq          # 예측값 ŷ = Ax̂
residual = b - pred             # 잔차 e = b - Ax̂
sse      = np.sum(residual ** 2)   # 오차 제곱합 S = Σeᵢ²

print("잔차      :", np.round(residual, 3))
print("잔차 합   :", round(residual.sum(), 10), "← 0이어야 정상")
print("오차제곱합:", round(sse, 4))

# ── 직교 조건 검증: Aᵀe = 0 이어야 함 (유도 ①단계) ─────────
print("Aᵀe (≈0?) :", np.round(A.T @ residual, 10))

# ── 예측해보기 ────────────────────────────────────────────
new_height = 168
print(f"키 {new_height}cm 예측 몸무게: {a_hat * new_height + b_hat:.1f}kg")

A shape: (5, 2) | b shape: (5,)
공식 방식 : [   1.16 -133.  ]
lstsq     : [   1.16 -133.  ]
찾은 직선: 몸무게 = 1.16 × 키 + (-133.00)
잔차      : [-0.4  0.8  0.  -0.8  0.4]
잔차 합   : 0.0 ← 0이어야 정상
오차제곱합: 1.6
Aᵀe (≈0?) : [1.e-10 0.e+00]
키 168cm 예측 몸무게: 61.9kg


## 🔗 함께 보기

- 📘 **책 (WikiDocs)** — [Zero to 머신러닝 딥러닝 Master](https://wikidocs.net/book/21464)
- 🏠 **홈페이지** — [mldict.net](https://mldict.net)
- ✍️ **기술 블로그** — [wikidocs.net/blog/@mldict](https://wikidocs.net/blog/@mldict/)
- 💻 **전체 코드 저장소** — [github.com/otoloism/Zero_to_ML_Master](https://github.com/otoloism/Zero_to_ML_Master)

👉 더 자세한 설명과 그림은 [책 원문](https://wikidocs.net/439841) 에서 확인하세요.